## PROCESAMEINTO Y LIMPIEZA  DE FUENTES
-  Autor: Germán Homero Morán Figueroa
- Descripción: Este notebook permite realizar la limpieza y depuración  de la fuente Fertilizations.csv, con el objetivo de consolidar bases limpias para la contrucción de la vista minable que sirve de insumo para los modelo de Machine Learning
-  Salida: Dataframe eventos_cultivo.csv y dataframeCilima.csv, se almacenan en Oro.

In [3]:
# Librerias y dependencias
# =========================================================================
import pandas as pd
import numpy as np
from unidecode import  unidecode
import plotly.express as px
import re
pd.options.display.max_columns = None

In [4]:
# Lectura del dataframe Fertilizations.csv
# =========================================================================================================
fertilizaciones = pd.read_csv("../../Data/Bronze/Fertilizations_3_Maiz_limpia.csv",sep=',',on_bad_lines='skip')
print("Longitud: ", fertilizaciones.shape)
fertilizaciones.head(5)

Longitud:  (7512, 12)


,ID_EVENTO,ID_PROD,FECHA_FERT,ID_FER_QUI,TIPO_FERTILIZACION,CANTIDAD_PROD_FERTI,PROD_QUI,PROD_ORG,PROD_ENM,N,P,K
0,24,40,2/06/2013,14,Enmiendas,1000,NaN,NaN,Cal,NaN,NaN,NaN
1,615,539,10/07/2013,624,Enmiendas,200,NaN,NaN,Cal,NaN,NaN,NaN
2,955,156,3/31/2014,1270,Enmiendas,20,NaN,NaN,Escorias,NaN,NaN,NaN
3,955,156,3/31/2014,1270,Enmiendas,40,NaN,NaN,Yeso,NaN,NaN,NaN
4,1213,1072,5/22/2014,1658,Enmiendas,750,NaN,NaN,Cal,NaN,NaN,NaN


In [10]:
# Lectura del dataframe eventos cordoba.
# ================================================================================
evento_cordoba = pd.read_csv("../../Data/Silver/eventos_cordoba_2017.csv")
Lista_IDEvento_Unicos_cordoba = list (evento_cordoba.ID_EVENTO.unique())
print("Cantidad de eventos unicos del departamento de cordoba: ", len(Lista_IDEvento_Unicos_cordoba))

Cantidad de eventos unicos del departamento de cordoba:  882


In [43]:

# Lectura del dataframe eventosParaFechas y eventos_finales obtenidos con la union de información de lso controles
# =================================================================================================================
eventosParaFechas = pd.read_csv("../../Data/Silver/cordoba_fechas_stages.csv")
eventosFinales = pd.read_csv("../../Data/Silver/eventos_controls.csv")


In [6]:
# Columnas archivo fertilizaciones
fertilizaciones.columns

Index(['ID_EVENTO', 'ID_PROD', 'FECHA_FERT', 'ID_FER_QUI',
       'TIPO_FERTILIZACION', 'CANTIDAD_PROD_FERTI', 'PROD_QUI', 'PROD_ORG',
       'PROD_ENM', 'N', 'P', 'K'],
      dtype='object')

In [7]:
# Analisis general del dataset
fertilizaciones.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7512 entries, 0 to 7511
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID_EVENTO            7512 non-null   int64  
 1   ID_PROD              7512 non-null   int64  
 2   FECHA_FERT           7512 non-null   object 
 3   ID_FER_QUI           7512 non-null   int64  
 4   TIPO_FERTILIZACION   7512 non-null   object 
 5   CANTIDAD_PROD_FERTI  7512 non-null   int64  
 6   PROD_QUI             7372 non-null   object 
 7   PROD_ORG             112 non-null    object 
 8   PROD_ENM             28 non-null     object 
 9   N                    7319 non-null   float64
 10  P                    7287 non-null   float64
 11  K                    7304 non-null   float64
dtypes: float64(3), int64(4), object(5)
memory usage: 704.4+ KB


In [8]:
# Analisis de nulos de dataset
fertilizaciones.isnull().sum()

ID_EVENTO                 0
ID_PROD                   0
FECHA_FERT                0
ID_FER_QUI                0
TIPO_FERTILIZACION        0
CANTIDAD_PROD_FERTI       0
PROD_QUI                140
PROD_ORG               7400
PROD_ENM               7484
N                       193
P                       225
K                       208
dtype: int64

In [9]:
# Analisis de variable tipo de fertilizacion.
# =================================================
df = px.data.tips()
fig = px.histogram(fertilizaciones, x="TIPO_FERTILIZACION")
fig.show()


*Nota:* La mayoria de fertilizaciones son quimicas , los otros tipos: Enmiendas y Organica tienen asociadas muy pocos registros.

In [11]:
# Se filtra unicamente fertilización correspondiantes al departamento de cordoba
# =================================================================================================
fertilizaciones = fertilizaciones[fertilizaciones.ID_EVENTO.isin(Lista_IDEvento_Unicos_cordoba)]
print("Cantidad de registros de Fertilizaciones asociadas a los eventos de cultivo referentes a Cordoba al ID Evento: ", len(fertilizaciones))

Cantidad de registros de Fertilizaciones asociadas a los eventos de cultivo referentes a Cordoba al ID Evento:  2258


In [12]:
# Validad que la columna CANTIDAD_PROD_FERTI sea numerica
fertilizaciones.CANTIDAD_PROD_FERTI

32        200
45         50
56      12000
57      12000
58      12000
        ...  
7498      100
7499      100
7500      100
7508      250
7509      125
Name: CANTIDAD_PROD_FERTI, Length: 2258, dtype: int64

In [13]:
# Eliminar los registros duplicados del dataset | en caso de que existan
# ======================================================================================
fertilizaciones = fertilizaciones.drop_duplicates()
print("Cantidad de registros despues de eliminar los duplicados: ", fertilizaciones.shape)


Cantidad de registros despues de eliminar los duplicados:  (2258, 12)


In [14]:
# Verificación del tipo de control
fertilizaciones.TIPO_FERTILIZACION.value_counts()


TIPO_FERTILIZACION
Quimica     2234
Organica      24
Name: count, dtype: int64

In [15]:
#  Se crea una nueva columna en el dataframe , deacuerdo al tipo de fertilización 
# ===========================================================================================
fertilizaciones["Fer"] = fertilizaciones.TIPO_FERTILIZACION.apply(lambda x: "CanFer"+x[0:3])
fertilizaciones.head(3)

,ID_EVENTO,ID_PROD,FECHA_FERT,ID_FER_QUI,TIPO_FERTILIZACION,CANTIDAD_PROD_FERTI,PROD_QUI,PROD_ORG,PROD_ENM,N,P,K,Fer
32,807,818,5/27/2014,1048,Organica,200,NaN,Lombri compost,NaN,NaN,NaN,NaN,CanFerOrg
45,2391,2289,6/04/2015,4270,Organica,50,NaN,Otro,NaN,NaN,NaN,NaN,CanFerOrg
56,3145,2802,8/18/2015,6321,Organica,12000,NaN,Estiercol,NaN,NaN,NaN,NaN,CanFerOrg


In [16]:
# Cantidad de Fertilizaciones realizadas por cada tipo de Siembra
# ========================================================================================================
frecFer = pd.crosstab(index=fertilizaciones.ID_EVENTO, columns=fertilizaciones.Fer).reset_index(drop=False)
frecFer

Fer,ID_EVENTO,CanFerOrg,CanFerQui
0,53,0,1
1,54,0,2
2,56,0,1
3,57,0,1
4,273,0,1
...,...,...,...
874,4671,0,3
875,4672,0,4
876,4673,0,4
877,4674,0,4


In [17]:
# Se cambia el formato de cada uno de los nutrientes (N,P,K) |verificar
# En este caso ya se encuentran en el formato establecido
fertilizaciones.K.head(5)

32   NaN
45   NaN
56   NaN
57   NaN
58   NaN
Name: K, dtype: float64

In [18]:
# Se verifica que la variable Fecha de Fertilización efectivamente este en el formato DateTime
fertilizaciones.FECHA_FERT = pd.to_datetime(fertilizaciones.FECHA_FERT,infer_datetime_format=True)
fertilizaciones.FECHA_FERT.head(5)

C:\Users\germanm\AppData\Local\Temp\ipykernel_13544\2166305443.py:2: UserWarning:

The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.



32   2014-05-27
45   2015-06-04
56   2015-08-18
57   2015-08-21
58   2015-08-17
Name: FECHA_FERT, dtype: datetime64[ns]

In [21]:
# Se realiza un merge entre [Eventos para fechas y fertilizaciones | ID EVENTO]
fertilizacionesEnEventos = pd.merge(eventosParaFechas,fertilizaciones,how='left',on=['ID_EVENTO'])
print("Tamaño Eventos Fertilizaciones: ", fertilizacionesEnEventos.shape)
fertilizacionesEnEventos.head(5)

Tamaño Eventos Fertilizaciones:  (2261, 18)


,ID_EVENTO,ID_LOTE,FECHA_SIEMBRA,FECHA_EMERGENCIA,FECHA_COSECHA,FECHA_FLORACION,ID_PROD,FECHA_FERT,ID_FER_QUI,TIPO_FERTILIZACION,CANTIDAD_PROD_FERTI,PROD_QUI,PROD_ORG,PROD_ENM,N,P,K,Fer
0,53,40,2013-05-13,2013-05-18,2013-09-26,2013-07-20,13.0,2013-05-28,956.0,Quimica,100.0,Urea,NaN,NaN,46.0,0.0,0.0,CanFerQui
1,54,43,2013-05-02,2013-05-07,2013-09-11,2013-07-10,14.0,2013-05-20,146.0,Quimica,100.0,Urea,NaN,NaN,46.0,0.0,0.0,CanFerQui
2,54,43,2013-05-02,2013-05-07,2013-09-11,2013-07-10,14.0,2013-06-04,147.0,Quimica,100.0,Urea,NaN,NaN,46.0,0.0,0.0,CanFerQui
3,56,44,2013-05-12,2013-05-17,2013-09-19,2013-07-15,15.0,2013-06-04,36.0,Quimica,100.0,Urea,NaN,NaN,46.0,0.0,0.0,CanFerQui
4,57,45,2013-05-07,2013-05-12,2013-09-12,2013-07-15,16.0,2013-05-22,148.0,Quimica,100.0,Urea,NaN,NaN,46.0,0.0,0.0,CanFerQui


In [25]:
# Conversión formato fechas a tipos datetime
# =====================================================
fertilizacionesEnEventos.FECHA_SIEMBRA =  pd.to_datetime(fertilizacionesEnEventos.FECHA_SIEMBRA,infer_datetime_format=True)
fertilizacionesEnEventos.FECHA_EMERGENCIA =  pd.to_datetime(fertilizacionesEnEventos.FECHA_EMERGENCIA,infer_datetime_format=True)
fertilizacionesEnEventos.FECHA_COSECHA =  pd.to_datetime(fertilizacionesEnEventos.FECHA_COSECHA,infer_datetime_format=True)
fertilizacionesEnEventos.FECHA_FLORACION =  pd.to_datetime(fertilizacionesEnEventos.FECHA_FLORACION,infer_datetime_format=True)


C:\Users\germanm\AppData\Local\Temp\ipykernel_13544\916273716.py:3: UserWarning:

The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.

C:\Users\germanm\AppData\Local\Temp\ipykernel_13544\916273716.py:4: UserWarning:

The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.

C:\Users\germanm\AppData\Local\Temp\ipykernel_13544\916273716.py:5: UserWarning:

The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.

C

In [26]:
# Conteo especifico de la categoria FER
fertilizacionesEnEventos.Fer.value_counts()

Fer
CanFerQui    2234
CanFerOrg      24
Name: count, dtype: int64

In [27]:
'''
Funcion (Fertilizaciones) para c/u de las columnas [organica Quimica]
Conteo del Numero de fertilizaciónes [Organicas o quimicas] realizadas en c/d fase del cultivo
En este caso solo se tienen en ceunta las siguientes fases:
 - Antes de la siembra
 - Siembra a emergencia
 - Emergencia a Floración

No se tienen en ceunta las fertilizaciones de floración- cosecha , dado que estas estas fases
el cultivo ya no tiene fertilizaciones registradas

Entrada: x: DataFrame 

'''

# Para las fertilizaciones Organicas
# ====================================================================================
def funcFertOrg1(x):
    if (x.FECHA_FERT < x.FECHA_SIEMBRA) and x.Fer == "CanFerOrg":
        return 1
    else:
        return 0

def funcFertOrg2(x):
    if (x.FECHA_FERT < x.FECHA_EMERGENCIA) and (x.FECHA_FERT  >= x.FECHA_SIEMBRA) and x.Fer=="CanFerOrg":
        return 1
    else:
        return 0

def funcFertOrg3(x):
    if (x.FECHA_FERT <= x.FECHA_COSECHA) and (x.FECHA_FERT  >= x.FECHA_EMERGENCIA) and (x.Fer == "CanFerOrg"):
        return 1
    else:
        return 0


# Para las Fetilizaciones Quimicas
# =====================================================================================================
def funcFertQui1(x):
    if (x.FECHA_FERT < x.FECHA_SIEMBRA) and x.Fer == "CanFerQui":
        return 1
    else:
        return 0


def funcFertQui2(x):
    if (x.FECHA_FERT < x.FECHA_EMERGENCIA) and (x.FECHA_FERT >= x.FECHA_SIEMBRA) and x.Fer=="CanFerQui":
        return 1
    else:
        return 0


def funcFertQui3(x):
    if (x.FECHA_FERT <= x.FECHA_COSECHA) and (x.FECHA_FERT  >= x.FECHA_EMERGENCIA) and (x.Fer == "CanFerQui"):
        return 1
    else:
        return 0

In [28]:
# Fertilizaciones Organicas para cada una de las etapas
# ================================================================================
fertilizacionesEnEventos["FerOrg_Antes_Siem"] = fertilizacionesEnEventos.apply(funcFertOrg1,axis=1)
fertilizacionesEnEventos["FerOrg_Siem_Emer"] = fertilizacionesEnEventos.apply(funcFertOrg2,axis=1)
fertilizacionesEnEventos["FerOrg_Emer_Flor"] = fertilizacionesEnEventos.apply(funcFertOrg3,axis=1)

In [29]:
# Fertilizaciones Quimicas para cada una de las etapas
# =================================================================================
fertilizacionesEnEventos["FerQui_Antes_Siem"] = fertilizacionesEnEventos.apply(funcFertQui1,axis=1)
fertilizacionesEnEventos["FerQui_Siem_Emer"] = fertilizacionesEnEventos.apply(funcFertQui2,axis=1)
fertilizacionesEnEventos["FerQui_Emer_Flor"] = fertilizacionesEnEventos.apply(funcFertQui3,axis=1)


In [30]:
fertilizacionesEnEventos

,ID_EVENTO,ID_LOTE,FECHA_SIEMBRA,FECHA_EMERGENCIA,FECHA_COSECHA,FECHA_FLORACION,ID_PROD,FECHA_FERT,ID_FER_QUI,TIPO_FERTILIZACION,CANTIDAD_PROD_FERTI,PROD_QUI,PROD_ORG,PROD_ENM,N,P,K,Fer,FerOrg_Antes_Siem,FerOrg_Siem_Emer,FerOrg_Emer_Flor,FerQui_Antes_Siem,FerQui_Siem_Emer,FerQui_Emer_Flor
0,53,40,2013-05-13,2013-05-18,2013-09-26,2013-07-20,13.0,2013-05-28,956.0,Quimica,100.0,Urea,NaN,NaN,46.0,0.0,0.0,CanFerQui,0,0,0,0,0,1
1,54,43,2013-05-02,2013-05-07,2013-09-11,2013-07-10,14.0,2013-05-20,146.0,Quimica,100.0,Urea,NaN,NaN,46.0,0.0,0.0,CanFerQui,0,0,0,0,0,1
2,54,43,2013-05-02,2013-05-07,2013-09-11,2013-07-10,14.0,2013-06-04,147.0,Quimica,100.0,Urea,NaN,NaN,46.0,0.0,0.0,CanFerQui,0,0,0,0,0,1
3,56,44,2013-05-12,2013-05-17,2013-09-19,2013-07-15,15.0,2013-06-04,36.0,Quimica,100.0,Urea,NaN,NaN,46.0,0.0,0.0,CanFerQui,0,0,0,0,0,1
4,57,45,2013-05-07,2013-05-12,2013-09-12,2013-07-15,16.0,2013-05-22,148.0,Quimica,100.0,Urea,NaN,NaN,46.0,0.0,0.0,CanFerQui,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2256,4674,4320,2016-05-09,2016-05-15,2016-09-21,2016-07-03,4784.0,2016-06-16,9522.0,Quimica,50.0,Urea,NaN,NaN,46.0,0.0,0.0,CanFerQui,0,0,0,0,0,1
2257,4674,4320,2016-05-09,2016-05-15,2016-09-21,2016-07-03,4784.0,2016-06-28,9523.0,Quimica,50.0,Urea,NaN,NaN,46.0,0.0,0.0,CanFerQui,0,0,0,0,0,1
2258,4675,4319,2016-05-09,2016-05-14,2016-09-21,2016-07-05,4784.0,2016-06-14,9525.0,Quimica,50.0,KCl,NaN,NaN,0.0,0.0,60.0,CanFerQui,0,0,0,0,0,1
2259,4675,4319,2016-05-09,2016-05-14,2016-09-21,2016-07-05,4784.0,2016-05-30,9524.0,Quimica,100.0,Urea,NaN,NaN,46.0,0.0,0.0,CanFerQui,0,0,0,0,0,1


In [31]:
'''
- La cantidad de Fertilizaciones [Quimica , Organica] asociada a cada uno de los lotes viene en kg/Ha 

'''
# Cantidad de fertilizaciones [ORGANICA Y QUIMICA] aplicadas en c/u de las etapas del cultivo.
# ================================================================================================
columnasFertilizaciones = ['ID_EVENTO','FerOrg_Antes_Siem','FerOrg_Siem_Emer', 'FerOrg_Emer_Flor'
                            ,'FerQui_Antes_Siem', 'FerQui_Siem_Emer', 'FerQui_Emer_Flor'] 

fertilizationByStages = fertilizacionesEnEventos[columnasFertilizaciones]
fertilizacionesPorEtapa = fertilizationByStages.groupby(["ID_EVENTO"]).sum().reset_index(drop=False)
fertilizacionesPorEtapa

,ID_EVENTO,FerOrg_Antes_Siem,FerOrg_Siem_Emer,FerOrg_Emer_Flor,FerQui_Antes_Siem,FerQui_Siem_Emer,FerQui_Emer_Flor
0,53,0,0,0,0,0,1
1,54,0,0,0,0,0,2
2,56,0,0,0,0,0,1
3,57,0,0,0,0,0,1
4,273,0,0,0,0,0,1
...,...,...,...,...,...,...,...
877,4671,0,0,0,0,0,3
878,4672,0,0,0,0,0,4
879,4673,0,0,0,0,0,4
880,4674,0,0,0,0,0,4


In [32]:
fertilizaciones.columns

Index(['ID_EVENTO', 'ID_PROD', 'FECHA_FERT', 'ID_FER_QUI',
       'TIPO_FERTILIZACION', 'CANTIDAD_PROD_FERTI', 'PROD_QUI', 'PROD_ORG',
       'PROD_ENM', 'N', 'P', 'K', 'Fer'],
      dtype='object')

In [33]:
fertilizaciones.CANTIDAD_PROD_FERTI

32        200
45         50
56      12000
57      12000
58      12000
        ...  
7498      100
7499      100
7500      100
7508      250
7509      125
Name: CANTIDAD_PROD_FERTI, Length: 2258, dtype: int64

In [34]:

'''
 Cantidad de fertilizaciones quimicas [N,P,K] realizadas en c/u de las etapas del cultivo
 [Siembra -Emergencia-Floración -Cosecha]
'''
# Funciones para el nitrogeno (N)
# ===================================================================================
def funcN1(x):
    if (x.FECHA_FERT <  x.FECHA_SIEMBRA) and (x.Fer == "CanFerQui"):
        op = (x.N/100)*(x.CANTIDAD_PROD_FERTI)
        return (op)
    else:
        return 0

def funcN2(x):
    if (x.FECHA_FERT < x.FECHA_EMERGENCIA) and (x.FECHA_FERT  >= x.FECHA_SIEMBRA) and (x.Fer == "CanFerQui"):
        op = (x.N/100)*(x.CANTIDAD_PROD_FERTI)
        return op
    else:
        return 0


def funcN3(x):
    if (x.FECHA_FERT <= x.FECHA_COSECHA) and (x.FECHA_FERT  >= x.FECHA_EMERGENCIA) and (x.Fer == "CanFerQui"):
        op = (x.N/100)*(x.CANTIDAD_PROD_FERTI)
        return op
    else:
        return 0

# Funciones para el Fosforo (P)
# ===================================================================================

def funcP1(x):
    if (x.FECHA_FERT < x.FECHA_SIEMBRA) and (x.Fer == "CanFerQui"):
        op = (x.P/100)*(x.CANTIDAD_PROD_FERTI)
        return op
    else:
        return 0

def funcP2(x):
    if (x.FECHA_FERT < x.FECHA_EMERGENCIA) and (x.FECHA_FERT  >= x.FECHA_SIEMBRA) and (x.Fer == "CanFerQui"):
        op = (x.P/100)*(x.CANTIDAD_PROD_FERTI)
        return op
    else:
        return 0

def funcP3(x):
    if (x.FECHA_FERT <= x.FECHA_COSECHA) and (x.FECHA_FERT  >= x.FECHA_EMERGENCIA) and (x.Fer == "CanFerQui"):
        op = (x.P/100)*(x.CANTIDAD_PROD_FERTI)
        return op
    else:
        return 0


# Funciones para el poteacio (K)
# ============================================================================================================0

def funcK1(x):
    if (x.FECHA_FERT < x.FECHA_SIEMBRA) and (x.Fer == "CanFerQui"):
        op = (x.K/100)*(x.CANTIDAD_PROD_FERTI)
        return op
    else:
        return 0 

def funck2(x):
    if (x.FECHA_FERT < x.FECHA_EMERGENCIA) and (x.FECHA_FERT  >= x.FECHA_SIEMBRA) and (x.Fer == "CanFerQui"):
        op = op = (x.K/100)*(x.CANTIDAD_PROD_FERTI)
        return op
    else:
        return 0

def funcK3(x):
    if (x.FECHA_FERT <= x.FECHA_COSECHA) and (x.FECHA_FERT  >= x.FECHA_EMERGENCIA) and (x.Fer == "CanFerQui"):
        op = op = (x.K/100)*(x.CANTIDAD_PROD_FERTI)
        return op
    else:
        return 0


In [35]:

# calculo de la cantidad de [n,p,k] en cada una de las etapas del cultivo
# =======================================================================================
fertilizacionesEnEventos['TotN_Antes_Siem'] = fertilizacionesEnEventos.apply(funcN1,axis=1)
fertilizacionesEnEventos['TotN_Siem_Emer'] = fertilizacionesEnEventos.apply(funcN2,axis=1)
fertilizacionesEnEventos['TotN_Emer_Flor'] = fertilizacionesEnEventos.apply(funcN3,axis=1)
fertilizacionesEnEventos['TotP_Antes_Siem'] = fertilizacionesEnEventos.apply(funcP1,axis=1)
fertilizacionesEnEventos['TotP_Siem_Emer'] = fertilizacionesEnEventos.apply(funcP2,axis=1)
fertilizacionesEnEventos['TotP_Emer_Flor'] = fertilizacionesEnEventos.apply(funcP3,axis=1)
fertilizacionesEnEventos['TotK_Antes_Siem'] = fertilizacionesEnEventos.apply(funcK1, axis=1)
fertilizacionesEnEventos['TotK_Siem_Emer'] = fertilizacionesEnEventos.apply(funck2,axis=1)
fertilizacionesEnEventos['TotK_Emer_Flor'] = fertilizacionesEnEventos.apply(funcK3,axis=1)

In [39]:
# Columnas nuevas 
# =====================================================================================
columnasFertilizacionQuimicasNPK = ['ID_EVENTO','TotN_Antes_Siem','TotN_Siem_Emer', 'TotN_Emer_Flor',
                                    'TotP_Antes_Siem', 'TotP_Siem_Emer','TotP_Emer_Flor',
                                    'TotK_Antes_Siem', 'TotK_Siem_Emer','TotK_Emer_Flor']

In [40]:

# Generacion nuevo datframe con als fertilizaciones x etpapa para cada evento
# =======================================================================================
fertbySrage = fertilizacionesEnEventos[columnasFertilizacionQuimicasNPK]
fertilizacionesQimicasPorEtapa = fertbySrage.groupby(["ID_EVENTO"]).sum().reset_index(drop=False)
print("Dimensiones: ", fertilizacionesQimicasPorEtapa.shape)
fertilizacionesQimicasPorEtapa.head(20)

Dimensiones:  (882, 10)


,ID_EVENTO,TotN_Antes_Siem,TotN_Siem_Emer,TotN_Emer_Flor,TotP_Antes_Siem,TotP_Siem_Emer,TotP_Emer_Flor,TotK_Antes_Siem,TotK_Siem_Emer,TotK_Emer_Flor
0,53,0.0,0.0,46.0,0.0,0.0,0.0,0.0,0.0,0.0
1,54,0.0,0.0,92.0,0.0,0.0,0.0,0.0,0.0,0.0
2,56,0.0,0.0,46.0,0.0,0.0,0.0,0.0,0.0,0.0
3,57,0.0,0.0,46.0,0.0,0.0,0.0,0.0,0.0,0.0
4,273,0.0,0.0,46.0,0.0,0.0,0.0,0.0,0.0,0.0
5,282,0.0,0.0,92.0,0.0,0.0,0.0,0.0,0.0,0.0
6,283,0.0,0.0,46.0,0.0,0.0,0.0,0.0,0.0,0.0
7,284,0.0,0.0,92.0,0.0,0.0,0.0,0.0,0.0,0.0
8,286,0.0,0.0,92.0,0.0,0.0,0.0,0.0,0.0,0.0
9,287,0.0,0.0,46.0,0.0,0.0,0.0,0.0,0.0,0.0


In [41]:
# Analisis de frecuencia de cada uno de los productos Organicos
fertilizaciones.PROD_ORG.value_counts()

PROD_ORG
Estiercol         22
Lombri compost     1
Otro               1
Name: count, dtype: int64

In [42]:


''' 
Finalmente se crea un unico dataframe que involucre:
- Cantidad de fertilizaciones quimicas [Kg/Ha] de [N,P,K] aplicado al cultivo en cada fase
-  Numero de Fertilizaciones [Organica , Quimicas]. Aplicada en cada etapa del cultivo
'''
VariablesDeFertilizaciones = pd.merge(fertilizacionesQimicasPorEtapa,fertilizacionesPorEtapa , how='inner')
print("Dimensiones: ", VariablesDeFertilizaciones.shape)
VariablesDeFertilizaciones.head(5)

Dimensiones:  (882, 16)


,ID_EVENTO,TotN_Antes_Siem,TotN_Siem_Emer,TotN_Emer_Flor,TotP_Antes_Siem,TotP_Siem_Emer,TotP_Emer_Flor,TotK_Antes_Siem,TotK_Siem_Emer,TotK_Emer_Flor,FerOrg_Antes_Siem,FerOrg_Siem_Emer,FerOrg_Emer_Flor,FerQui_Antes_Siem,FerQui_Siem_Emer,FerQui_Emer_Flor
0,53,0.0,0.0,46.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,1
1,54,0.0,0.0,92.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,2
2,56,0.0,0.0,46.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,1
3,57,0.0,0.0,46.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,1
4,273,0.0,0.0,46.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,1


In [44]:
# Finalmente se realiza el merge con los eventos fianales y se incluye la parte de Fertilizaciones
# ==================================================================================================
eventosFinales = pd.merge(eventosFinales,VariablesDeFertilizaciones, how='inner')
print(eventosFinales.shape)

(882, 47)


In [ ]:
# Se gurda los datos en Silver
# Este dataframe contienen información de Controles, Fertilizaciones e información General
# ===========================================================================================
eventosFinales.to_csv("../../Data/Silver/eventos_Fertilizations.csv", index=False)